# Ensemble Models for Time Series Forecasting
Since ARIMA and SARIMA didn't work well due to sparse, sporadic data, I'll experiment with ensemble models for better performance.
1. Random Forest (Bagging)
- Strengths: Simple, works well with default settings, easy to use.
- Weaknesses: Struggles with sparse data and complex patterns compared to boosting methods.
2. XGBoost (Extreme Gradient Boosting)
- Strengths: High performance on structured data, better for imbalanced data, more control over parameters.
- Weaknesses: Prone to overfitting, computationally expensive.
3. LightGBM (Light Gradient Boosting Machine)
- Strengths: Faster on large datasets, efficient with memory, better for sparse datasets.
- Weaknesses: Prone to overfitting (leaf-wise splitting), requires careful tuning.

### Data Preparation

#### Load Datasets

In [1]:
import pandas as pd
import numpy as np

# Load datasets
sales = pd.read_csv('m5-forecasting-accuracy/sales_train_validation.csv')
calendar = pd.read_csv('m5-forecasting-accuracy/calendar.csv')
sell_price = pd.read_csv('m5-forecasting-accuracy/sell_prices.csv')

In [2]:
sales.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,3,0,1,1,1,3,0,1,1
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,2,1,1,1,0,1,1,1
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,5,4,1,0,1,3,7,2
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,1,0,1,1,2,2,2,4


In [3]:
sum(sales.isnull().sum())

0

In [4]:
calendar.head()

,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1


In [5]:
sell_price.head()

,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


In [6]:
sum(sell_price.isnull().sum())

0

In [7]:
print("Sales Data Types:")
print(sales.dtypes)


Sales Data Types:
id          object
item_id     object
dept_id     object
cat_id      object
store_id    object
             ...  
d_1909       int64
d_1910       int64
d_1911       int64
d_1912       int64
d_1913       int64
Length: 1919, dtype: object


In [8]:
print("\nCalendar Data Types:")
print(calendar.dtypes)

print("\nPrice Data Types:")
print(sell_price.dtypes)


Calendar Data Types:
date            object
wm_yr_wk         int64
weekday         object
wday             int64
month            int64
year             int64
d               object
event_name_1    object
event_type_1    object
event_name_2    object
event_type_2    object
snap_CA          int64
snap_TX          int64
snap_WI          int64
dtype: object

Price Data Types:
store_id       object
item_id        object
wm_yr_wk        int64
sell_price    float64
dtype: object


#### Feature Engineering
1. Date feature (weekday, month, year,...)
2. Lag feature (take sales from the previous week/month into consideration)
3. Encoding features (store_id, dept_id, item_id) into numerical format

In [9]:
# change the data type
def optimize_numeric_types(df):
    for col in df.select_dtypes(include=['int', 'float']).columns:
        if str(df[col].dtype).startswith('int'):
            df[col] = pd.to_numeric(df[col], downcast='integer')
        elif str(df[col].dtype).startswith('float'):
            df[col] = pd.to_numeric(df[col], downcast='float')
    return df

def convert_object_columns(df):
    for col in df.select_dtypes(include='object').columns:
        # date columns: object -> datetime
        if 'date' in col:
            df[col] = pd.to_datetime(df[col])
        # Convert obejct to category if not datetime
        elif df[col].nunique() < 0.5 * len(df):  # Only categorize if cardinality is low
            df[col] = df[col].astype('category')
    return df

# Optimize Calendar
calendar_data = optimize_numeric_types(calendar)
calendar_data = convert_object_columns(calendar)

# Optimize Price
price_data = optimize_numeric_types(sell_price)
price_data = convert_object_columns(sell_price)

# Optimize Sales
sales_data = optimize_numeric_types(sales)
sales_data = convert_object_columns(sales)

print("Sales Data Types:")
print(sales.dtypes)

print("\nCalendar Data Types:")
print(calendar.dtypes)

print("\nPrice Data Types:")
print(sell_price.dtypes)

Sales Data Types:
id            object
item_id     category
dept_id     category
cat_id      category
store_id    category
              ...   
d_1909          int8
d_1910          int8
d_1911         int16
d_1912         int16
d_1913         int16
Length: 1919, dtype: object

Calendar Data Types:
date            datetime64[ns]
wm_yr_wk                 int16
weekday               category
wday                      int8
month                     int8
year                     int16
d                       object
event_name_1          category
event_type_1          category
event_name_2          category
event_type_2          category
snap_CA                   int8
snap_TX                   int8
snap_WI                   int8
dtype: object

Price Data Types:
store_id      category
item_id       category
wm_yr_wk         int16
sell_price     float32
dtype: object


In [10]:
#convert categorical columns in calendar dataset to numerical codes while ensuring no negative values
for col in calendar.columns:
    if calendar[col].dtype == 'category':
        calendar[col] = calendar[col].cat.codes.astype('int16')
        calendar[col] -= calendar[col].min()

  #convert categorical columns in price dataset to numerical codes while ensuring no negative values
for col in sell_price.columns:
    if sell_price[col].dtype == 'category':
        sell_price[col] = sell_price[col].cat.codes.astype('int16')
        sell_price[col] -= sell_price[col].min()      

In [11]:
#convert categorical columns in sales dataset to numerical codes while ensuring no negative values
categorical_columns = ['id','item_id', 'dept_id', 'cat_id', 'store_id','state_id']
for col in categorical_columns:
     if col != 'id':
          sales[col] = sales[col].cat.codes.astype('int16')
          sales[col] -= sales[col].min()

# Unpivot sales data
sales_data_long = pd.melt(sales, 
                          id_vars=categorical_columns, 
                          value_vars=[col for col in sales.columns if col.startswith('d_')], 
                          var_name='d', 
                          value_name='sales')

# Merge with calendar data
merged_data = sales_data_long.merge(calendar, on='d', how='left', copy=False)

# Merge with price data (on store_id, item_id, and wm_yr_wk)
merged_data = merged_data.merge(sell_price, on=['store_id', 'item_id', 'wm_yr_wk'], how='left', copy=False)


In [12]:
merged_data.head(20)

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,HOBBIES_1_001_CA_1_validation,1437,3,1,0,0,d_1,0,2011-01-29,11101,...,1,2011,0,0,0,0,0,0,0,NaN
1,HOBBIES_1_002_CA_1_validation,1438,3,1,0,0,d_1,0,2011-01-29,11101,...,1,2011,0,0,0,0,0,0,0,NaN
2,HOBBIES_1_003_CA_1_validation,1439,3,1,0,0,d_1,0,2011-01-29,11101,...,1,2011,0,0,0,0,0,0,0,NaN
3,HOBBIES_1_004_CA_1_validation,1440,3,1,0,0,d_1,0,2011-01-29,11101,...,1,2011,0,0,0,0,0,0,0,NaN
4,HOBBIES_1_005_CA_1_validation,1441,3,1,0,0,d_1,0,2011-01-29,11101,...,1,2011,0,0,0,0,0,0,0,NaN
5,HOBBIES_1_006_CA_1_validation,1442,3,1,0,0,d_1,0,2011-01-29,11101,...,1,2011,0,0,0,0,0,0,0,NaN
6,HOBBIES_1_007_CA_1_validation,1443,3,1,0,0,d_1,0,2011-01-29,11101,...,1,2011,0,0,0,0,0,0,0,NaN
7,HOBBIES_1_008_CA_1_validation,1444,3,1,0,0,d_1,12,2011-01-29,11101,...,1,2011,0,0,0,0,0,0,0,0.46
8,HOBBIES_1_009_CA_1_validation,1445,3,1,0,0,d_1,2,2011-01-29,11101,...,1,2011,0,0,0,0,0,0,0,1.56
9,HOBBIES_1_010_CA_1_validation,1446,3,1,0,0,d_1,0,2011-01-29,11101,...,1,2011,0,0,0,0,0,0,0,3.17


In [13]:
print(merged_data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58327370 entries, 0 to 58327369
Data columns (total 22 columns):
 #   Column        Dtype         
---  ------        -----         
 0   id            object        
 1   item_id       int16         
 2   dept_id       int16         
 3   cat_id        int16         
 4   store_id      int16         
 5   state_id      int16         
 6   d             object        
 7   sales         int16         
 8   date          datetime64[ns]
 9   wm_yr_wk      int16         
 10  weekday       int16         
 11  wday          int8          
 12  month         int8          
 13  year          int16         
 14  event_name_1  int16         
 15  event_type_1  int16         
 16  event_name_2  int16         
 17  event_type_2  int16         
 18  snap_CA       int8          
 19  snap_TX       int8          
 20  snap_WI       int8          
 21  sell_price    float32       
dtypes: datetime64[ns](1), float32(1), int16(13), int8(5), object(2)


##### New Features

In [14]:
def add_date_features(df, date_col='date'):
    df = df.copy()
    df['mday'] = df[date_col].dt.day.astype('int8') #the day of the month
    df['quarter'] = df[date_col].dt.quarter.astype('int8')
    #df['day_of_year'] = df[date_col].dt.dayofyear.astype('int16')
    df['is_weekend'] = df[date_col].dt.weekday.isin([5, 6]).astype('int8')  # 5=Saturday, 6=Sunday
    #df['is_month_start'] = df[date_col].dt.is_month_start.astype('int8')
    #df['is_month_end'] = df[date_col].dt.is_month_end.astype('int8')
    return df

def add_lag_feature(df, lags=[7, 28]):
    df = df.sort_values(by=['id', 'date'])  
    for lag in lags:
        df[f'lag_{lag}'] = df.groupby('id')['sales'].shift(lag)
    return df

In [15]:
merged_data = add_date_features(merged_data)
merged_data = add_lag_feature(merged_data, lags=[7, 28])

In [16]:
merged_data.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,event_type_2,snap_CA,snap_TX,snap_WI,sell_price,mday,quarter,is_weekend,lag_7,lag_28
1612,FOODS_1_001_CA_1_validation,0,0,0,0,0,d_1,3,2011-01-29,11101,...,0,0,0,0,2.0,29,1,1,NaN,NaN
32102,FOODS_1_001_CA_1_validation,0,0,0,0,0,d_2,0,2011-01-30,11101,...,0,0,0,0,2.0,30,1,1,NaN,NaN
62592,FOODS_1_001_CA_1_validation,0,0,0,0,0,d_3,0,2011-01-31,11101,...,0,0,0,0,2.0,31,1,0,NaN,NaN
93082,FOODS_1_001_CA_1_validation,0,0,0,0,0,d_4,1,2011-02-01,11101,...,0,1,1,0,2.0,1,1,0,NaN,NaN
123572,FOODS_1_001_CA_1_validation,0,0,0,0,0,d_5,4,2011-02-02,11101,...,0,1,0,1,2.0,2,1,0,NaN,NaN


In [17]:
# to train the model, I'll drop rows where any lag is NaN
merged_data = merged_data.dropna(subset=['lag_7', 'lag_28'])

In [18]:
merged_data.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,event_type_2,snap_CA,snap_TX,snap_WI,sell_price,mday,quarter,is_weekend,lag_7,lag_28
855332,FOODS_1_001_CA_1_validation,0,0,0,0,0,d_29,2,2011-02-26,11105,...,0,0,0,0,2.0,26,1,1,1.0,3.0
885822,FOODS_1_001_CA_1_validation,0,0,0,0,0,d_30,2,2011-02-27,11105,...,0,0,0,0,2.0,27,1,1,2.0,0.0
916312,FOODS_1_001_CA_1_validation,0,0,0,0,0,d_31,0,2011-02-28,11105,...,0,0,0,0,2.0,28,1,0,0.0,0.0
946802,FOODS_1_001_CA_1_validation,0,0,0,0,0,d_32,2,2011-03-01,11105,...,0,1,1,0,2.0,1,1,0,2.0,1.0
977292,FOODS_1_001_CA_1_validation,0,0,0,0,0,d_33,1,2011-03-02,11105,...,0,1,0,1,2.0,2,1,0,2.0,4.0


In [19]:
print(merged_data.info())

<class 'pandas.core.frame.DataFrame'>
Index: 57473650 entries, 855332 to 58325932
Data columns (total 27 columns):
 #   Column        Dtype         
---  ------        -----         
 0   id            object        
 1   item_id       int16         
 2   dept_id       int16         
 3   cat_id        int16         
 4   store_id      int16         
 5   state_id      int16         
 6   d             object        
 7   sales         int16         
 8   date          datetime64[ns]
 9   wm_yr_wk      int16         
 10  weekday       int16         
 11  wday          int8          
 12  month         int8          
 13  year          int16         
 14  event_name_1  int16         
 15  event_type_1  int16         
 16  event_name_2  int16         
 17  event_type_2  int16         
 18  snap_CA       int8          
 19  snap_TX       int8          
 20  snap_WI       int8          
 21  sell_price    float32       
 22  mday          int8          
 23  quarter       int8          
 

In [20]:
merged_data['sales'].describe()

count    5.747365e+07
mean     1.130270e+00
std      3.870784e+00
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      1.000000e+00
max      7.630000e+02
Name: sales, dtype: float64

The sales data range from 0 to 763, with the mean around 1.13 and a standard deviation of 3.87. This shows that most of the sales values are quite small, but there are occasional large spikes.

### Model Setup and Training
#### LightGBM Key Parameters Reference

I researched LightGBM and summarized some useful parameters for future reference:
- objective: Defines the learning task.  
  - examples: 'regression', 'binary', 'multiclass', 'poisson' (useful for count data like sales).
- metric: Evaluation metric used to monitor performance.  
  - examples: rmse, mae, logloss, auc, multi_logloss.
- learning_rate: Controls the step size during training. 越小训练越慢但可能更稳
- num_leaves: Maximum number of leaves in one tree. 单棵树的最大叶子节点数，越大模型越复杂
- min_data_in_leaf: Minimum number of samples required in a leaf. Larger values reduce overfitting.
- feature_fraction: 
- bagging_fraction / sub_row: Fraction of data to sample for each tree (row-wise sampling).
- bagging_freq: Frequency (in iterations) to perform bagging.
- lambda_l1/lambda_l2: L1/L2 regularization
- force_row_wise: Forces LightGBM to use row-wise histogram building (useful when feature count is large).
- num_iterations: Maximum number of boosting rounds.
- early_stopping_rounds: Stops training if validation metric does not improve after N rounds.
- verbosity: -1 静音，0 提示，1 全部打印
- boosting_type: gbdt, dart, goss, rf, ...


In [21]:
# split into train and validation sets (28 days)
max_train_day = merged_data['date'].max() - pd.Timedelta(days=28)
train_df = merged_data[merged_data['date'] <= max_train_day]
valid_df = merged_data[merged_data['date'] > max_train_day]

features = [col for col in merged_data.columns if col not in ['id', 'sales', 'date', 'd']]
target = 'sales'

X_train = train_df[features]
y_train = train_df[target]
X_valid = valid_df[features]
y_valid = valid_df[target]
print(X_train.shape, y_train.shape)
print(X_valid.shape,y_valid.shape)

(56619930, 23) (56619930,)
(853720, 23) (853720,)


In [22]:
import lightgbm as lgb

train_data = lgb.Dataset(X_train, label = y_train)
valid_data = lgb.Dataset(X_valid, label = y_valid, reference=train_data)

reg_params = {
    'objective': 'regression',
    'metric': 'rmse', # measure prediction error
    'force_row_wise': True, # more memory-efficient on large datasets
    'boosting_type': 'gbdt',
    'num_leaves': 128, # limits tree complexity
    'min_data_in_leaf': 512,
    'verbosity': 1
}

In [23]:
callbacks = [lgb.early_stopping(stopping_rounds=50)]

reg_model = lgb.train(
    reg_params,
    train_set = train_data,
    valid_sets = [train_data, valid_data],
    num_boost_round = 1000,
    callbacks=callbacks,
)

[LightGBM] [Info] Total Bins 1074
[LightGBM] [Info] Number of data points in the train set: 56619930, number of used features: 23
[LightGBM] [Info] Start training from score 1.126407
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[240]	training's rmse: 2.25809	valid_1's rmse: 2.16079


In [32]:
reg_model.save_model("reg_model.lgb")

In [24]:
poi_params = {
    'objective': 'poisson',
    'metric': 'rmse', # measure prediction error
    'force_row_wise': True, # more memory-efficient on large datasets
    'boosting_type': 'gbdt',
    'num_leaves': 128, # limits tree complexity
    'min_data_in_leaf': 512,
    'verbosity': 1
}

In [29]:
callbacks = [lgb.early_stopping(stopping_rounds=50)]

poi_model = lgb.train(
    poi_params,
    train_set = train_data,
    valid_sets = [train_data, valid_data],
    num_boost_round = 1000,
    callbacks=callbacks,
)

[LightGBM] [Info] Total Bins 1074
[LightGBM] [Info] Number of data points in the train set: 56619930, number of used features: 23
[LightGBM] [Info] Start training from score 0.119033
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[100]	training's rmse: 2.43541	valid_1's rmse: 2.19933


In [31]:
poi_model.save_model("poi_model.lgb")

### Tuning (Hyperparameter Optimization)

### Model Predictions

In [ ]:
def create_test_df(merged_data, calendar, last_date, forecast_days=28, lags=[7, 28]):
    """
    build a test set for forecasting future sales over the next 28 days.
    
    parameters:
    - merged_data: historical sales data with features
    - calendar: the original calendar.csv data
    - last_date: the last date in merged_data (e.g., pd.to_datetime('2024-04-24'))
    - forecast_days: number of days to forecast, default is 28
    - lags
    
    returns:
    - test_df
    """
    future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=forecast_days)
    #all_ids = merged_data['id'].unique()

    # get static columns
    static_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
    static_df = merged_data[static_cols].drop_duplicates()

    test_list = []
    for date in future_dates:
        temp = static_df.copy()
        temp['date'] = date
        test_list.append(temp)

    test_df = pd.concat(test_list, axis=0)

    # Merge with calendar to get date-related features
    cal = calendar.copy()
    cal['date'] = pd.to_datetime(cal['date'])
    test_df = test_df.merge(cal, on='date', how='left')

    # Merge with sell_price
    test_df = test_df.merge(sell_price, on=['store_id', 'item_id', 'wm_yr_wk'], how='left')

    # Add additional custom date features
    test_df = add_date_features(test_df, date_col='date')

    # Combine with historical data to allow lag feature generation
    full_df = pd.concat([merged_data, test_df], sort=False)

    # Add lag features
    full_df = add_lag_feature(full_df, lags=lags)

    # Extract only the rows for future dates
    test_df = full_df[full_df['date'].isin(future_dates)].copy()

    return test_df


In [38]:
# the sales_validation set ends on 2016-04-24
last_train_date = pd.to_datetime('2016-04-24')
test_df = create_test_df(merged_data, calendar, last_train_date)

In [39]:
print(test_df.shape)

(853720, 27)


In [40]:
test_df.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,event_type_2,snap_CA,snap_TX,snap_WI,sell_price,mday,quarter,is_weekend,lag_7,lag_28
0,FOODS_1_001_CA_1_validation,0,0,0,0,0,d_1914,NaN,2016-04-25,11613,...,0,0,0,0,2.24,25,2,0,4.0,2.0
30490,FOODS_1_001_CA_1_validation,0,0,0,0,0,d_1915,NaN,2016-04-26,11613,...,0,0,0,0,2.24,26,2,0,1.0,1.0
60980,FOODS_1_001_CA_1_validation,0,0,0,0,0,d_1916,NaN,2016-04-27,11613,...,0,0,0,0,2.24,27,2,0,1.0,1.0
91470,FOODS_1_001_CA_1_validation,0,0,0,0,0,d_1917,NaN,2016-04-28,11613,...,0,0,0,0,2.24,28,2,0,0.0,0.0
121960,FOODS_1_001_CA_1_validation,0,0,0,0,0,d_1918,NaN,2016-04-29,11613,...,0,0,0,0,2.24,29,2,0,1.0,4.0


In [41]:
features = [col for col in merged_data.columns if col not in ['id', 'sales', 'date', 'd']]
X_test = test_df[features]

In [42]:
X_test

,item_id,dept_id,cat_id,store_id,state_id,wm_yr_wk,weekday,wday,month,year,...,event_type_2,snap_CA,snap_TX,snap_WI,sell_price,mday,quarter,is_weekend,lag_7,lag_28
0,0,0,0,0,0,11613,1,3,4,2016,...,0,0,0,0,2.24,25,2,0,4.0,2.0
30490,0,0,0,0,0,11613,5,4,4,2016,...,0,0,0,0,2.24,26,2,0,1.0,1.0
60980,0,0,0,0,0,11613,6,5,4,2016,...,0,0,0,0,2.24,27,2,0,1.0,1.0
91470,0,0,0,0,0,11613,4,6,4,2016,...,0,0,0,0,2.24,28,2,0,0.0,0.0
121960,0,0,0,0,0,11613,0,7,4,2016,...,0,0,0,0,2.24,29,2,0,1.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
731759,3048,6,2,9,2,11616,6,5,5,2016,...,0,0,0,0,5.94,18,2,0,NaN,0.0
762249,3048,6,2,9,2,11616,4,6,5,2016,...,0,0,0,0,5.94,19,2,0,NaN,0.0
792739,3048,6,2,9,2,11616,0,7,5,2016,...,0,0,0,0,5.94,20,2,0,NaN,0.0
823229,3048,6,2,9,2,11617,2,1,5,2016,...,0,0,0,0,5.94,21,2,1,NaN,0.0


In [43]:
test_df['sales_pred'] = reg_model.predict(X_test)

In [ ]:
result = test_df[['id', 'date', 'sales_pred']]
#result.to_csv('reg_lgb_prediction.csv', index=False)

### Evaluation

In [53]:
sales_eval = pd.read_csv('m5-forecasting-accuracy/sales_train_validation.csv')

In [ ]:
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

def evaluate_predictions(test_df, sales_eval_wide, plot_examples=3):
    """
    Evaluate predictions using test_df with 'id', 'd', 'sales_pred' columns,
    compared to ground truth from sales_train_evaluation.csv.

    Parameters:
    - test_df: DataFrame with ['id', 'd', 'sales_pred']
    - sales_eval_wide: Wide-format ground truth data (sales_train_evaluation.csv)
    - plot_examples: Number of random prediction plots to show (default=3)

    Returns:
    - merged: DataFrame with merged predictions and actuals
    """
    # Melt wide format to long format
    sales_eval_long = sales_eval_wide.melt(id_vars=['id'], var_name='d', value_name='sales_actual')

    # Merge on id and d
    merged = test_df.merge(sales_eval_long, on=['id', 'd'], how='left')
    print(merged['sales_actual'].isna().sum())
    
    merged['error'] = merged['sales_pred'] - merged['sales_actual']
    rmse = np.sqrt(mean_squared_error(merged['sales_actual'], merged['sales_pred']))
    print(f"RMSE over forecast period: {rmse:.4f}")
    return merged


In [61]:
result_df = evaluate_predictions(test_df, sales_eval)

853720


ValueError: Input contains NaN.